In [1]:
# scripts/04_centrality.ipynb

import sys
import torch
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# 1. Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading MPNet on {device}...")
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

# 2. Test Article (A tricky one)
# This text has a clear "intro" (usually important) and "details" (less important).
text_sentences = [
    # -- Core Theme --
    "SpaceX successfully launched the Starship rocket today.",
    "The launch marks a major milestone in space exploration.",
    "Elon Musk declared the mission a total success.",
    
    # -- Specific Details (Should have LOWER centrality) --
    "The rocket lifted off at 8:00 AM EST.",
    "It burned 5,000 tons of liquid methane.",
    "The weather was clear with light winds.",
    "Spectators gathered three miles away to watch.",
    
    # -- Tangential/Noise (Should have LOWEST centrality) --
    "I forgot to bring my binoculars.",
    "Coffee was served at the viewing deck."
]

print(f"Analyzing {len(text_sentences)} sentences...\n")

# 3. Generate Embeddings & Cosine Matrix
embeddings = model.encode(text_sentences, convert_to_tensor=True)
cos_scores = util.cos_sim(embeddings, embeddings)

# Move to CPU for math
cos_scores = cos_scores.cpu().numpy()

# 4. Method A: Degree Centrality (Simple Sum)
# Logic: Sum of similarities to all other sentences.
# "How much do I look like everyone else?"
degree_scores = np.sum(cos_scores, axis=1)
# Normalize 0-1
degree_scores = (degree_scores - degree_scores.min()) / (degree_scores.max() - degree_scores.min())

# 5. Method B: Eigenvector Centrality (LexRank / Power Iteration)
# Logic: "I am important if I am similar to other IMPORTANT sentences."
# This is usually better for catching the "theme" recursively.
try:
    # We use the adjacency matrix (cosine scores)
    # Ensure no negative values (Cosine can be -1 to 1, but usually 0-1 for text)
    adj_matrix = np.maximum(cos_scores, 0)
    
    # Compute Eigenvalues/vectors
    eigenvalues, eigenvectors = np.linalg.eig(adj_matrix)
    
    # The principal eigenvector (corresponding to largest eigenvalue) is the centrality
    # We take the real part (numpy returns complex)
    eigen_scores = np.abs(eigenvectors[:, 0])
    
    # Normalize 0-1
    eigen_scores = (eigen_scores - eigen_scores.min()) / (eigen_scores.max() - eigen_scores.min())
    
except Exception as e:
    print(f"Eigenvector calculation failed: {e}")
    eigen_scores = np.zeros(len(text_sentences))

# 6. Compare Results
results = []
for i, sent in enumerate(text_sentences):
    results.append({
        "Sentence": sent,
        "Degree (Sum)": round(degree_scores[i], 3),
        "Eigen (PageRank)": round(eigen_scores[i], 3)
    })

df = pd.DataFrame(results).sort_values(by="Eigen (PageRank)", ascending=False)

print("--- Top Ranked Sentences (Summary Candidates) ---")
display(df)

# 7. Recommendation Check
top_sentence = df.iloc[0]["Sentence"]
if "SpaceX" in top_sentence or "milestone" in top_sentence:
    print("\nSUCCESS: The model correctly identified the main theme.")
else:
    print("\nFAILURE: The model prioritized a detail sentence.")

/opt/conda/envs/sentinel-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading MPNet on cpu...
Analyzing 9 sentences...

--- Top Ranked Sentences (Summary Candidates) ---


,Sentence,Degree (Sum),Eigen (PageRank)
1,The launch marks a major milestone in space ex...,1.000,1.000
3,The rocket lifted off at 8:00 AM EST.,0.918,0.924
2,Elon Musk declared the mission a total success.,0.841,0.869
0,SpaceX successfully launched the Starship rock...,0.569,0.780
4,"It burned 5,000 tons of liquid methane.",0.422,0.453
6,Spectators gathered three miles away to watch.,0.111,0.142
5,The weather was clear with light winds.,0.044,0.138
8,Coffee was served at the viewing deck.,0.005,0.018
7,I forgot to bring my binoculars.,0.000,0.000



SUCCESS: The model correctly identified the main theme.
